# N-BEATS Model Experiment

This notebook trains an N-BEATS forecasting model for the Walmart weekly sales dataset. It follows the full experiment flow: environment setup, data loading, time-aware validation split, preprocessing/window creation, W&B logging, model training, validation diagnostics, and best-model registration.

## 1. Environment setup

In [ ]:
%pip install -q "torch>=2.3,<3" "wandb>=0.19,<1" "pandas>=2.2,<3" "numpy>=1.26,<3" "matplotlib>=3.8,<4"

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    print('Google Drive mount skipped. This is expected outside Colab.')

In [ ]:
import json
import math
import os
import platform
from itertools import product
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import wandb

SEED = 42
WANDB_ENTITY = "kende23-n-a"
WANDB_PROJECT = "Walmart-Recruiting---Store-Sales-Forecasting"
WANDB_GROUP = "nbeats-experiments"

DATA_DIR_CANDIDATES = [
    Path('/content/drive/MyDrive/walmart_competition_data'),
    Path('/content/drive/My Drive/walmart_competition_data'),
    Path('/content/walmart_competition_data'),
    Path('../../data'),
    Path('data'),
]

OUTPUT_DIR = Path('/content/artifacts/nbeats') if Path('/content').exists() else Path('artifacts/nbeats')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CONFIG = {
    'validation_weeks': 32,
    'context_length': 52,
    'forecast_horizon': 32,
    'min_series_length': 84,
    'batch_size': 128,
    'max_epochs': 30,
    'learning_rate': 1e-3,
    'weight_decay': 1e-4,
    'hidden_units': 256,
    'num_blocks': 4,
    'num_layers': 4,
    'dropout': 0.10,
    'holiday_weight': 5.0,
    'clip_grad_norm': 1.0,
    'num_workers': 2,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
}


def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True


seed_everything(SEED)
print(f"Using device: {CONFIG['device']}")

## 2. Load and validate data

N-BEATS is a univariate deep-learning forecaster, so the core input is the weekly sales history for each `(Store, Dept)` series. We still load all competition files because they are useful for validation, future inference, and experiment metadata.

In [ ]:
def resolve_data_dir(candidates):
    for candidate in candidates:
        if candidate.exists() and (candidate / 'train.csv').exists():
            return candidate
    searched = '\n'.join(str(path) for path in candidates)
    raise FileNotFoundError(f'Could not find train.csv. Searched:\n{searched}')


DATA_DIR = resolve_data_dir(DATA_DIR_CANDIDATES)
print(f'Data directory: {DATA_DIR}')

train_raw = pd.read_csv(DATA_DIR / 'train.csv', parse_dates=['Date'])
test_raw = pd.read_csv(DATA_DIR / 'test.csv', parse_dates=['Date'])
features_raw = pd.read_csv(DATA_DIR / 'features.csv', parse_dates=['Date'])
stores_raw = pd.read_csv(DATA_DIR / 'stores.csv')

required_train_cols = {'Store', 'Dept', 'Date', 'Weekly_Sales', 'IsHoliday'}
missing_cols = required_train_cols.difference(train_raw.columns)
if missing_cols:
    raise ValueError(f'train.csv is missing required columns: {sorted(missing_cols)}')

print('train shape:', train_raw.shape)
print('test shape:', test_raw.shape)
print('features shape:', features_raw.shape)
print('stores shape:', stores_raw.shape)
print('date range:', train_raw['Date'].min().date(), 'to', train_raw['Date'].max().date())

## 3. Preprocessing and feature engineering for N-BEATS

For N-BEATS the most important feature engineering is converting tabular rows into clean time-series tensors:

- sort observations by date,
- create one complete weekly index,
- pivot each `(Store, Dept)` into one sales series,
- fill occasional missing weeks inside a series,
- apply a time-aware validation split using the last 32 weeks,
- transform sales with `log1p` for stable neural-network training,
- normalize each series using only training-period statistics,
- create sliding windows of 52 historical weeks to predict the next 32 weeks.

In [ ]:
train_df = train_raw.copy()
train_df['Store'] = train_df['Store'].astype(int)
train_df['Dept'] = train_df['Dept'].astype(int)
train_df = train_df.sort_values(['Store', 'Dept', 'Date']).reset_index(drop=True)

all_dates = pd.date_range(train_df['Date'].min(), train_df['Date'].max(), freq='W-FRI')
sales_pivot = (
    train_df
    .pivot_table(index=['Store', 'Dept'], columns='Date', values='Weekly_Sales', aggfunc='sum')
    .reindex(columns=all_dates)
    .sort_index()
)

holiday_by_date = (
    train_df.groupby('Date')['IsHoliday']
    .max()
    .reindex(all_dates)
    .fillna(False)
    .astype(bool)
)

min_required_weeks = CONFIG['context_length'] + CONFIG['forecast_horizon']
valid_series_mask = sales_pivot.notna().sum(axis=1) >= min_required_weeks
sales_pivot = sales_pivot.loc[valid_series_mask]

# Fill only after removing very short series. This keeps the time grid regular for neural training.
filled_sales = sales_pivot.ffill(axis=1).bfill(axis=1).fillna(0.0)
series_index = filled_sales.index
series_keys = pd.DataFrame(series_index.tolist(), columns=['Store', 'Dept'])

validation_weeks = CONFIG['validation_weeks']
context_length = CONFIG['context_length']
forecast_horizon = CONFIG['forecast_horizon']

if validation_weeks != forecast_horizon:
    raise ValueError('This notebook expects validation_weeks to equal forecast_horizon for direct multi-step validation.')

validation_start = len(all_dates) - validation_weeks
if validation_start < context_length:
    raise ValueError('Not enough historical weeks before validation. Reduce context_length or validation_weeks.')

train_dates = all_dates[:validation_start]
validation_dates = all_dates[validation_start:]

sales_values_original = filled_sales.to_numpy(dtype=np.float32)
# Neural networks train more stably on log sales. Negative sales are rare return/refund rows, so only the model target transform is clipped.
log_sales_values = np.log1p(np.clip(sales_values_original, a_min=0.0, a_max=None)).astype(np.float32)

series_center = log_sales_values[:, :validation_start].mean(axis=1, keepdims=True)
series_scale = log_sales_values[:, :validation_start].std(axis=1, keepdims=True)
series_scale = np.where(series_scale < 1e-6, 1.0, series_scale).astype(np.float32)
normalized_values = ((log_sales_values - series_center) / series_scale).astype(np.float32)

print('number of weekly dates:', len(all_dates))
print('number of usable Store-Dept series:', len(series_index))
print('training date range:', train_dates.min().date(), 'to', train_dates.max().date())
print('validation date range:', validation_dates.min().date(), 'to', validation_dates.max().date())
print('sales tensor shape:', normalized_values.shape)

In [ ]:
def make_training_windows(values: np.ndarray, context: int, horizon: int, validation_start_idx: int):
    X_windows = []
    y_windows = []
    series_ids = []
    last_window_start = validation_start_idx - context - horizon
    if last_window_start < 0:
        raise ValueError('Not enough history to create training windows before validation.')

    for series_id in range(values.shape[0]):
        for start in range(last_window_start + 1):
            split = start + context
            X_windows.append(values[series_id, start:split])
            y_windows.append(values[series_id, split:split + horizon])
            series_ids.append(series_id)

    return (
        np.asarray(X_windows, dtype=np.float32),
        np.asarray(y_windows, dtype=np.float32),
        np.asarray(series_ids, dtype=np.int32),
    )


X_train, y_train, train_series_ids = make_training_windows(
    normalized_values,
    context_length,
    forecast_horizon,
    validation_start,
)

X_val = normalized_values[:, validation_start - context_length:validation_start].astype(np.float32)
y_val_normalized = normalized_values[:, validation_start:validation_start + forecast_horizon].astype(np.float32)
y_val_original = sales_values_original[:, validation_start:validation_start + forecast_horizon].astype(np.float32)
val_holiday_flags = holiday_by_date.loc[validation_dates].to_numpy(dtype=bool)

print('X_train:', X_train.shape)
print('y_train:', y_train.shape)
print('X_val:', X_val.shape)
print('y_val:', y_val_original.shape)
print('validation holiday weeks:', int(val_holiday_flags.sum()))

## 4. Log preprocessing and feature engineering to W&B

In [ ]:
wandb.login(key=os.environ.get('WANDB_API_KEY'), relogin=False)

preprocessing_run = wandb.init(
    entity=WANDB_ENTITY,
    project=WANDB_PROJECT,
    group=WANDB_GROUP,
    job_type='preprocessing',
    name='NBEATS_Preprocessing_and_Windowing',
    config={
        **CONFIG,
        'seed': SEED,
        'model_family': 'N-BEATS',
        'data_dir': str(DATA_DIR),
        'train_rows': int(len(train_raw)),
        'test_rows': int(len(test_raw)),
        'usable_series': int(len(series_index)),
        'train_windows': int(len(X_train)),
        'total_weeks': int(len(all_dates)),
        'train_start_date': str(train_dates.min().date()),
        'train_end_date': str(train_dates.max().date()),
        'validation_start_date': str(validation_dates.min().date()),
        'validation_end_date': str(validation_dates.max().date()),
        'target_transform': 'clip_negative_then_log1p',
        'series_scaling': 'per_series_standardization_fit_on_training_period',
    },
)

summary_table = wandb.Table(
    columns=['step', 'description'],
    data=[
        ['cleaning', 'Sorted by Store, Dept, Date and validated required columns.'],
        ['regular_time_grid', 'Created one weekly Friday index for every Store-Dept series.'],
        ['missing_weeks', 'Forward/backward filled internal missing weeks after removing short series.'],
        ['validation_split', 'Reserved the last 32 weeks as validation to avoid time leakage.'],
        ['target_transform', 'Used log1p on non-negative sales for stable neural training.'],
        ['scaling', 'Normalized each series with training-period mean and standard deviation.'],
        ['windowing', 'Created 52-week input windows with 32-week forecasting targets.'],
    ],
)
preprocessing_run.log({
    'preprocessing/summary': summary_table,
    'preprocessing/usable_series': len(series_index),
    'preprocessing/train_windows': len(X_train),
})
preprocessing_run.finish()

## 5. Dataset, model, and metric definitions

In [ ]:
class WindowDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray | None = None):
        self.X = torch.from_numpy(X.astype(np.float32))
        self.y = None if y is None else torch.from_numpy(y.astype(np.float32))

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        if self.y is None:
            return self.X[idx]
        return self.X[idx], self.y[idx]


class NBeatsBlock(nn.Module):
    def __init__(self, input_size: int, horizon: int, hidden_units: int, num_layers: int, dropout: float):
        super().__init__()
        layers = []
        current_size = input_size
        for _ in range(num_layers):
            layers.append(nn.Linear(current_size, hidden_units))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            current_size = hidden_units
        self.net = nn.Sequential(*layers)
        self.backcast_head = nn.Linear(hidden_units, input_size)
        self.forecast_head = nn.Linear(hidden_units, horizon)

    def forward(self, x):
        hidden = self.net(x)
        backcast = self.backcast_head(hidden)
        forecast = self.forecast_head(hidden)
        return backcast, forecast


class NBeats(nn.Module):
    def __init__(self, input_size: int, horizon: int, hidden_units: int, num_blocks: int, num_layers: int, dropout: float):
        super().__init__()
        self.horizon = horizon
        self.blocks = nn.ModuleList([
            NBeatsBlock(input_size, horizon, hidden_units, num_layers, dropout)
            for _ in range(num_blocks)
        ])

    def forward(self, x):
        residual = x
        forecast = torch.zeros(x.size(0), self.horizon, device=x.device)
        for block in self.blocks:
            backcast, block_forecast = block(residual)
            residual = residual - backcast
            forecast = forecast + block_forecast
        return forecast


def inverse_transform_predictions(pred_normalized: np.ndarray, center: np.ndarray, scale: np.ndarray) -> np.ndarray:
    pred_log = pred_normalized * scale + center
    pred_sales = np.expm1(pred_log)
    return np.clip(pred_sales, a_min=0.0, a_max=None)


def weighted_mae(y_true: np.ndarray, y_pred: np.ndarray, holiday_flags: np.ndarray, holiday_weight: float = 5.0) -> float:
    weights = np.where(holiday_flags.reshape(1, -1), holiday_weight, 1.0)
    weights = np.broadcast_to(weights, y_true.shape)
    return float(np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights))


def mae(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.mean(np.abs(y_true - y_pred)))


def rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

## 6. Baseline validation score

The baseline repeats the most recent 32 observed weeks from the 52-week context. This gives a simple seasonal reference before training N-BEATS.

In [ ]:
seasonal_naive_pred = sales_values_original[:, validation_start - forecast_horizon:validation_start]
baseline_wmae = weighted_mae(y_val_original, seasonal_naive_pred, val_holiday_flags, CONFIG['holiday_weight'])
baseline_mae = mae(y_val_original, seasonal_naive_pred)

print(f'Seasonal naive validation WMAE: {baseline_wmae:.4f}')
print(f'Seasonal naive validation MAE: {baseline_mae:.4f}')

## 7. Hyperparameter brute force and N-BEATS training

This section runs a full grid search. Every hyperparameter combination trains for `30` epochs, logs its own W&B run, and updates the saved best model only when it improves validation Weighted MAE.


In [ ]:
def evaluate_model(model: nn.Module, X: np.ndarray, batch_size: int = 512):
    model.eval()
    predictions = []
    loader = DataLoader(WindowDataset(X), batch_size=batch_size, shuffle=False, num_workers=0)
    with torch.no_grad():
        for batch_X in loader:
            batch_X = batch_X.to(CONFIG['device'])
            batch_pred = model(batch_X).cpu().numpy()
            predictions.append(batch_pred)
    return np.concatenate(predictions, axis=0)


HYPERPARAMETER_GRID = {
    'batch_size': [32, 64, 128],
    'learning_rate': [1e-3, 1e-2, 1e-1],
    'hidden_units': [128, 256],
    'num_blocks': [3, 4],
    'num_layers': [3, 4],
    'dropout': [0.0, 0.10],
    'weight_decay': [0.0, 1e-4],
}

# Set this to a small integer while debugging if needed. Keep None for the full brute-force run.
MAX_GRID_RUNS = None


def build_grid_configs(grid: dict, max_runs: int | None = None):
    keys = list(grid.keys())
    values = [grid[key] for key in keys]
    configs = [dict(zip(keys, combination)) for combination in product(*values)]
    if max_runs is not None:
        configs = configs[:max_runs]
    return configs


def train_one_grid_config(trial_id: int, total_trials: int, trial_params: dict):
    trial_config = {**CONFIG, **trial_params, 'max_epochs': 30}
    seed_everything(SEED + trial_id)

    train_loader = DataLoader(
        WindowDataset(X_train, y_train),
        batch_size=trial_config['batch_size'],
        shuffle=True,
        num_workers=trial_config['num_workers'],
        pin_memory=trial_config['device'] == 'cuda',
    )

    model = NBeats(
        input_size=context_length,
        horizon=forecast_horizon,
        hidden_units=trial_config['hidden_units'],
        num_blocks=trial_config['num_blocks'],
        num_layers=trial_config['num_layers'],
        dropout=trial_config['dropout'],
    ).to(trial_config['device'])

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=trial_config['learning_rate'],
        weight_decay=trial_config['weight_decay'],
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
    criterion = nn.L1Loss()

    run_name = f"NBEATS_grid_trial_{trial_id:03d}"
    training_run = wandb.init(
        entity=WANDB_ENTITY,
        project=WANDB_PROJECT,
        group=WANDB_GROUP,
        job_type='training',
        name=run_name,
        config={
            **trial_config,
            'seed': SEED + trial_id,
            'trial_id': trial_id,
            'total_trials': total_trials,
            'model_family': 'N-BEATS',
            'search_strategy': 'brute_force_grid_search',
            'baseline_validation_weighted_mae': baseline_wmae,
            'train_windows': int(len(X_train)),
            'validation_series': int(len(X_val)),
        },
    )
    wandb.watch(model, log='gradients', log_freq=100)

    trial_best_wmae = math.inf
    trial_best_epoch = -1
    trial_best_state = None

    for epoch in range(1, trial_config['max_epochs'] + 1):
        model.train()
        train_losses = []

        for batch_X, batch_y in train_loader:
            batch_X = batch_X.to(trial_config['device'])
            batch_y = batch_y.to(trial_config['device'])

            optimizer.zero_grad(set_to_none=True)
            batch_pred = model(batch_X)
            loss = criterion(batch_pred, batch_y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), trial_config['clip_grad_norm'])
            optimizer.step()
            train_losses.append(loss.item())

        train_loss = float(np.mean(train_losses))
        val_pred_normalized = evaluate_model(model, X_val)
        val_pred_original = inverse_transform_predictions(val_pred_normalized, series_center, series_scale)

        validation_wmae = weighted_mae(y_val_original, val_pred_original, val_holiday_flags, trial_config['holiday_weight'])
        validation_mae = mae(y_val_original, val_pred_original)
        validation_rmse = rmse(y_val_original, val_pred_original)
        scheduler.step(validation_wmae)

        training_run.log({
            'epoch': epoch,
            'train/l1_log_scaled': train_loss,
            'validation/weighted_mae': validation_wmae,
            'validation/mae': validation_mae,
            'validation/rmse': validation_rmse,
            'learning_rate': optimizer.param_groups[0]['lr'],
        })

        if validation_wmae < trial_best_wmae:
            trial_best_wmae = validation_wmae
            trial_best_epoch = epoch
            trial_best_state = {key: value.detach().cpu() for key, value in model.state_dict().items()}

        print(
            f"Trial {trial_id + 1:03d}/{total_trials:03d} | epoch {epoch:03d}/030 | "
            f"batch={trial_config['batch_size']} lr={trial_config['learning_rate']} | "
            f"train L1={train_loss:.5f} val WMAE={validation_wmae:.4f}"
        )

    training_run.summary['best_epoch'] = trial_best_epoch
    training_run.summary['best_validation_weighted_mae'] = trial_best_wmae
    training_run.summary['baseline_validation_weighted_mae'] = baseline_wmae
    training_run.finish()

    return {
        'trial_id': trial_id,
        'run_name': run_name,
        'params': trial_params,
        'config': trial_config,
        'best_epoch': trial_best_epoch,
        'best_validation_weighted_mae': trial_best_wmae,
        'state_dict': trial_best_state,
    }


grid_configs = build_grid_configs(HYPERPARAMETER_GRID, MAX_GRID_RUNS)
print(f'Grid search combinations: {len(grid_configs)}')
print('Each combination trains for 30 epochs.')

best_validation_wmae = math.inf
best_epoch = -1
best_trial_id = -1
best_run_name = None
best_trial_params = None
best_trial_config = None
best_model_path = OUTPUT_DIR / 'nbeats_best_model.pt'
grid_results = []

for trial_id, trial_params in enumerate(grid_configs):
    result = train_one_grid_config(trial_id, len(grid_configs), trial_params)
    grid_results.append({
        'trial_id': result['trial_id'],
        'run_name': result['run_name'],
        'best_epoch': result['best_epoch'],
        'best_validation_weighted_mae': result['best_validation_weighted_mae'],
        **result['params'],
    })

    if result['best_validation_weighted_mae'] < best_validation_wmae:
        best_validation_wmae = result['best_validation_weighted_mae']
        best_epoch = result['best_epoch']
        best_trial_id = result['trial_id']
        best_run_name = result['run_name']
        best_trial_params = result['params']
        best_trial_config = result['config']
        torch.save(
            {
                'model_state_dict': result['state_dict'],
                'config': best_trial_config,
                'hyperparameters': best_trial_params,
                'series_center': series_center,
                'series_scale': series_scale,
                'series_index': series_keys.to_dict(orient='records'),
                'validation_dates': [str(date.date()) for date in validation_dates],
                'best_trial_id': best_trial_id,
                'best_run_name': best_run_name,
                'best_epoch': best_epoch,
                'best_validation_weighted_mae': best_validation_wmae,
            },
            best_model_path,
        )


grid_results_df = pd.DataFrame(grid_results).sort_values('best_validation_weighted_mae')
grid_results_path = OUTPUT_DIR / 'nbeats_grid_search_results.csv'
grid_results_df.to_csv(grid_results_path, index=False)

search_run = wandb.init(
    entity=WANDB_ENTITY,
    project=WANDB_PROJECT,
    group=WANDB_GROUP,
    job_type='hyperparameter_search_summary',
    name='NBEATS_Brute_Force_Grid_Search_Summary',
    config={
        **CONFIG,
        'search_strategy': 'brute_force_grid_search',
        'max_epochs_per_trial': 30,
        'total_trials': len(grid_configs),
        'hyperparameter_grid': HYPERPARAMETER_GRID,
    },
)
search_run.log({
    'grid_search/results': wandb.Table(dataframe=grid_results_df),
    'grid_search/best_validation_weighted_mae': best_validation_wmae,
    'grid_search/best_trial_id': best_trial_id,
})
search_run.summary['best_trial_id'] = best_trial_id
search_run.summary['best_run_name'] = best_run_name
search_run.summary['best_epoch'] = best_epoch
search_run.summary['best_validation_weighted_mae'] = best_validation_wmae
search_run.summary['best_hyperparameters'] = best_trial_params
search_run.finish()

print('--- N-BEATS brute-force hyperparameter search results ---')
print(grid_results_df.head(10))
print(f'Best trial id: {best_trial_id}')
print(f'Best run name: {best_run_name}')
print(f'Best epoch: {best_epoch}')
print(f'Best validation weighted MAE: {best_validation_wmae:.4f}')
print('Best hyperparameters:')
print(best_trial_params)

## 8. Validation diagnostics

In [ ]:
checkpoint = torch.load(best_model_path, map_location=CONFIG['device'])
model.load_state_dict(checkpoint['model_state_dict'])
model.to(CONFIG['device'])

best_val_pred_normalized = evaluate_model(model, X_val)
best_val_pred_original = inverse_transform_predictions(best_val_pred_normalized, series_center, series_scale)

prediction_rows = []
for series_id, (store, dept) in enumerate(series_index):
    for horizon_idx, date in enumerate(validation_dates):
        actual = float(y_val_original[series_id, horizon_idx])
        prediction = float(best_val_pred_original[series_id, horizon_idx])
        prediction_rows.append({
            'Store': int(store),
            'Dept': int(dept),
            'Date': date,
            'Weekly_Sales': actual,
            'Prediction': prediction,
            'IsHoliday': bool(val_holiday_flags[horizon_idx]),
            'AbsoluteError': abs(actual - prediction),
        })

val_predictions_df = pd.DataFrame(prediction_rows)
weekly_errors_df = (
    val_predictions_df
    .groupby('Date', as_index=False)
    .agg(Weekly_MAE=('AbsoluteError', 'mean'), IsHoliday=('IsHoliday', 'max'))
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(
    val_predictions_df['Weekly_Sales'],
    val_predictions_df['Prediction'],
    s=5,
    alpha=0.25,
)
axes[0].set_title('Actual vs predicted weekly sales')
axes[0].set_xlabel('Actual')
axes[0].set_ylabel('Prediction')

axes[1].plot(weekly_errors_df['Date'], weekly_errors_df['Weekly_MAE'], marker='o')
axes[1].set_title('Validation MAE by week')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('MAE')
axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

diagnostics_path = OUTPUT_DIR / 'validation_predictions.csv'
weekly_errors_path = OUTPUT_DIR / 'weekly_validation_errors.csv'
val_predictions_df.to_csv(diagnostics_path, index=False)
weekly_errors_df.to_csv(weekly_errors_path, index=False)

analysis_run = wandb.init(
    entity=WANDB_ENTITY,
    project=WANDB_PROJECT,
    group=WANDB_GROUP,
    job_type='evaluation',
    name='NBEATS_Validation_Diagnostics',
    config={
        **CONFIG,
        'best_epoch': best_epoch,
        'best_validation_weighted_mae': best_validation_wmae,
    },
)
analysis_run.log({
    'validation/actual_vs_prediction_and_weekly_error': wandb.Image(fig),
    'validation/predictions_sample': wandb.Table(dataframe=val_predictions_df.head(1000)),
    'validation/weekly_errors': wandb.Table(dataframe=weekly_errors_df),
})
analysis_run.finish()

## 9. Save and register the best N-BEATS model

This cell logs only the best model bundle. It does not rerun every trial or retrain multiple models. The artifact contains the PyTorch weights plus the preprocessing metadata required to rebuild the same inference tensors.

In [ ]:
metadata = {
    'model_family': 'N-BEATS',
    'best_trial_id': int(best_trial_id),
    'best_run_name': best_run_name,
    'best_epoch': int(best_epoch),
    'best_validation_weighted_mae': float(best_validation_wmae),
    'baseline_validation_weighted_mae': float(baseline_wmae),
    'context_length': int(context_length),
    'forecast_horizon': int(forecast_horizon),
    'validation_weeks': int(validation_weeks),
    'validation_dates': [str(date.date()) for date in validation_dates],
    'target_transform': 'clip_negative_then_log1p',
    'series_scaling': 'per_series_standardization_fit_on_training_period',
    'best_hyperparameters': best_trial_params,
    'search_strategy': 'brute_force_grid_search',
    'max_epochs_per_trial': 30,
    'registered_model_name': 'NBEATS-Best-Model',
    'python_version': platform.python_version(),
    'torch_version': torch.__version__,
}

metadata_path = OUTPUT_DIR / 'metadata.json'
config_path = OUTPUT_DIR / 'config.json'
series_index_path = OUTPUT_DIR / 'series_index.csv'

metadata_path.write_text(json.dumps(metadata, indent=2))
config_path.write_text(json.dumps(CONFIG, indent=2))
series_keys.to_csv(series_index_path, index=False)

registry_run = wandb.init(
    entity=WANDB_ENTITY,
    project=WANDB_PROJECT,
    group=WANDB_GROUP,
    job_type='model_registry',
    name='NBEATS_Register_Best_Model',
    config=metadata,
)

model_artifact = wandb.Artifact(
    name='nbeats-best-model',
    type='model',
    metadata=metadata,
    description='Best N-BEATS model bundle for Walmart Store Sales forecasting.',
)
model_artifact.add_file(str(best_model_path), name='nbeats_best_model.pt')
model_artifact.add_file(str(metadata_path), name='metadata.json')
model_artifact.add_file(str(config_path), name='config.json')
model_artifact.add_file(str(series_index_path), name='series_index.csv')
model_artifact.add_file(str(diagnostics_path), name='validation_predictions.csv')
model_artifact.add_file(str(weekly_errors_path), name='weekly_validation_errors.csv')
model_artifact.add_file(str(grid_results_path), name='nbeats_grid_search_results.csv')

logged_artifact = registry_run.log_artifact(model_artifact, aliases=['best', 'latest', f'epoch-{best_epoch}'])
registry_run.link_artifact(
    logged_artifact,
    target_path='wandb-registry-model/NBEATS-Best-Model',
    aliases=['best', 'latest', 'production-candidate'],
)

registry_run.summary['best_trial_id'] = best_trial_id
registry_run.summary['best_run_name'] = best_run_name
registry_run.summary['best_epoch'] = best_epoch
registry_run.summary['best_hyperparameters'] = best_trial_params
registry_run.summary['best_validation_weighted_mae'] = best_validation_wmae
registry_run.summary['registered_model_name'] = 'NBEATS-Best-Model'
registry_run.finish()

print('Logged and linked best N-BEATS model artifact to W&B Model Registry.')
print(f'Best model path: {best_model_path}')

## 10. Final notes

The notebook now performs brute-force grid search, trains every combination for 30 epochs, saves the best trial by validation Weighted MAE, and registers only that best model bundle. For future test-set forecasting, rebuild the `(Store, Dept)` weekly matrix with the same preprocessing steps, take the latest 52 weeks for each series, apply the saved per-series normalization, run the model, and inverse-transform the forecast.
